# Green vs. Grey: A Risk-Adjusted Growth Opportunity Analysis of India's Energy Sector
**Author:** Sramantika Sen

Compares a basket of renewable-focused Indian energy stocks against a basket of conventional/legacy power stocks using risk-adjusted return metrics, market benchmarking (Nifty 50), and a short-term ARIMA price forecast.

## Step 1: Install and import libraries

In [ ]:
!pip install yfinance statsmodels --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

## Step 2: Define the two baskets

In [ ]:
green_basket = ["ADANIGREEN.NS", "SUZLON.NS", "INOXWIND.NS"]   # renewable-focused
grey_basket  = ["NTPC.NS", "COALINDIA.NS", "TATAPOWER.NS"]     # conventional/legacy power
all_tickers = green_basket + grey_basket

## Step 3: Pull 5 years of price data

In [ ]:
data = yf.download(all_tickers, start="2020-01-01", end="2025-09-05")["Close"]
data = data.dropna()
data.head()

## Step 4: Compute daily returns

In [ ]:
returns = data.pct_change().dropna()

## Step 5: Per-company risk metrics (Return, Volatility, Sharpe, VaR)

In [ ]:
summary = pd.DataFrame(index=all_tickers)
summary["Annual Return %"] = returns.mean() * 252 * 100
summary["Annual Volatility %"] = returns.std() * np.sqrt(252) * 100
summary["Sharpe Ratio"] = (returns.mean()*252) / (returns.std()*np.sqrt(252))
summary["VaR 95% (daily)"] = returns.quantile(0.05)
print(summary)

## Step 6: Basket-level comparison (Green vs Grey)

In [ ]:
green_returns = returns[green_basket].mean(axis=1)
grey_returns  = returns[grey_basket].mean(axis=1)

for name, r in [("Green Basket", green_returns), ("Grey Basket", grey_returns)]:
    ann_ret = r.mean()*252
    ann_vol = r.std()*np.sqrt(252)
    sharpe  = ann_ret/ann_vol
    print(f"{name}: Return={ann_ret:.2%}, Volatility={ann_vol:.2%}, Sharpe={sharpe:.2f}")

## Step 7: Growth Opportunity Score (custom metric) — ranked

In [ ]:
summary["Growth Opportunity Score"] = summary["Annual Return %"] / summary["Annual Volatility %"]
print(summary.sort_values("Growth Opportunity Score", ascending=False))

## Step 8: Max Drawdown

In [ ]:
def max_drawdown(prices):
    cummax = prices.cummax()
    drawdown = (prices - cummax) / cummax
    return drawdown.min() * 100

for ticker in all_tickers:
    dd = max_drawdown(data[ticker])
    summary.loc[ticker, "Max Drawdown %"] = dd
    print(f"{ticker}: Max Drawdown = {dd:.2f}%")

## Step 9: Market benchmarking — Beta & Alpha vs Nifty 50

In [ ]:
nifty_raw = yf.download("^NSEI", start="2020-01-01", end="2025-09-05")["Close"]
nifty = nifty_raw.squeeze()
nifty_returns = nifty.pct_change().dropna()

aligned = returns.copy()
aligned["Nifty"] = nifty_returns

beta_alpha = {}
for ticker in all_tickers:
    valid = aligned[[ticker, "Nifty"]].dropna()
    cov = valid[ticker].cov(valid["Nifty"])
    var = valid["Nifty"].var()
    beta = cov / var
    alpha = (valid[ticker].mean()*252) - beta*(valid["Nifty"].mean()*252)
    beta_alpha[ticker] = {"Beta": beta, "Alpha (annual)": alpha}
    summary.loc[ticker, "Beta"] = beta
    summary.loc[ticker, "Alpha (annual)"] = alpha

beta_df = pd.DataFrame(beta_alpha).T
print(beta_df)

## Step 10: Growth Opportunity Score bar chart (colored by basket, with annotations)

In [ ]:
plt.figure(figsize=(9,6))
sorted_summary = summary.sort_values("Growth Opportunity Score", ascending=False)
colors = ["seagreen" if ticker in green_basket else "dimgray" for ticker in sorted_summary.index]
plt.bar(sorted_summary.index, sorted_summary["Growth Opportunity Score"], color=colors)
plt.title("Growth Opportunity Score by Company\n(Green = renewable-focused, Grey = conventional)")
plt.ylabel("Growth Opportunity Score")
plt.xticks(rotation=30, ha="right")

plt.annotate("Pure renewable \u2014\nyet lowest score",
             xy=("ADANIGREEN.NS", summary.loc["ADANIGREEN.NS", "Growth Opportunity Score"]),
             xytext=(0, 40), textcoords="offset points",
             ha="center", fontsize=9, color="darkred",
             arrowprops=dict(arrowstyle="->", color="darkred"))

plt.annotate("Legacy player \u2014\nyet near-top score",
             xy=("TATAPOWER.NS", summary.loc["TATAPOWER.NS", "Growth Opportunity Score"]),
             xytext=(0, 40), textcoords="offset points",
             ha="center", fontsize=9, color="darkblue",
             arrowprops=dict(arrowstyle="->", color="darkblue"))

plt.tight_layout()
plt.savefig("growth_opportunity_score_chart.png")
plt.show()

## Step 11: ARIMA 90-day forecast — Green Basket

In [ ]:
green_price = data[green_basket].mean(axis=1).reset_index(drop=True)
model_green = ARIMA(green_price, order=(5,1,0))
fit_green = model_green.fit()
forecast_green = fit_green.forecast(steps=90)

hist_len = len(green_price)
plt.figure()
plt.plot(range(hist_len - 180, hist_len), green_price[-180:].values, label="Historical")
plt.plot(range(hist_len, hist_len + 90), forecast_green, label="Forecast")
plt.legend()
plt.title("Green Basket: 90-Day Price Forecast")
plt.xlabel("Trading days")
plt.savefig("green_basket_forecast.png")
plt.show()

## Step 12: ARIMA 90-day forecast — Grey Basket

In [ ]:
grey_price = data[grey_basket].mean(axis=1).reset_index(drop=True)
model_grey = ARIMA(grey_price, order=(5,1,0))
fit_grey = model_grey.fit()
forecast_grey = fit_grey.forecast(steps=90)

hist_len = len(grey_price)
plt.figure()
plt.plot(range(hist_len - 180, hist_len), grey_price[-180:].values, label="Historical")
plt.plot(range(hist_len, hist_len + 90), forecast_grey, label="Forecast")
plt.legend()
plt.title("Grey Basket: 90-Day Price Forecast")
plt.xlabel("Trading days")
plt.savefig("grey_basket_forecast.png")
plt.show()

## Done
Three chart files are now saved in this Colab session's file storage (see the folder icon on the left sidebar): `growth_opportunity_score_chart.png`, `green_basket_forecast.png`, `grey_basket_forecast.png`. Download each one (right-click → Download) to upload alongside this notebook and the README to GitHub.